In [33]:
from __future__ import annotations
import os
import json
import importlib
import argparse
from functools import partial
from collections import defaultdict

import random
import numpy as np
import pickle

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import mcts_search_mini_v15
from core import mcts_search_mini_v55
from core import mcts_search_mini_v35
from core import mcts_search_mini_v45
from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k

# from core.llm_engine import rm_engine
# from core.llms import rm_generate
# import logging
# logging.basicConfig(format='%(message)s', level=logging.ERROR)

In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [7]:
stop

NameError: name 'stop' is not defined

In [4]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [5]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 7            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0

# mcts parameter
config.num_phases = 1
config.num_batches = 1
config.batch_budget = config.num_batches*config.max_depths 

config.lam = 10 
config.normalize_embeds = True

config.cpuct_root = 0
config.cpuct_leaf = 0
config.cpuct = 2
config.ds_beta = 1.0
config.ds_alpha = 10.0
config.use_ppl = True

config.version = "v51"



In [6]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
# num_questions = 1
print(f"num_questions = {num_questions}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 38
batch_of_questions = [data_by_levels[level][q_idx]['problem']]
batch_of_answers = [data_by_levels[level][q_idx]['answer']]
print(batch_of_questions)
print(batch_of_answers)

num_questions = 128
['The quadratic $x^2+(2.6)x+3.6$ can be written in the form $(x+b)^2+c$, where $b$ and $c$ are constants. What is $b+c$ (as a decimal)?']
['3.21']


In [15]:
trial_idx = 0
config.max_depths = 7
config_name = f"mcts--n-{config.n}--d-{config.max_depths}--normalize-{config.normalize_embeds}--level-{level}--qidx-{q_idx}"
print(config_name)
with open(f"results/tree_{config_name}.pkl", 'rb') as fout:
    tree_dict = pickle.load(fout)

mcts--n-4--d-7--normalize-True--level-4--trial-0--qidx-38


In [36]:

completions_dict_base = defaultdict()
for tag, output in tree_dict.items():
    if output["is_completed"]:
        completions_dict_base[tag] = output["text"]

config_name = f"mcts--base--n-{config.n}--d-{config.max_depths}--level-{level}--qidx-{q_idx}"
with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
    json.dump(completions_dict_base, fout)
    fout.write('\n')

In [ ]:
# print(tree_dict['0.0.0']['text'])
# print(tree_dict['0.0.1']['text'])
# print(tree_dict['0.0.2']['text'])
# print(tree_dict['0.0.3']['text'])
print(tree_dict['0.0.0.2.1.0.1.1'])
print(tree_dict['0.0.0.2.1.0.1.2'])
print(tree_dict['0.0.0.0.0.0.2'])
print(tree_dict['0.0.0.0.0.0.1.0'])

In [83]:
config.n = 4
config.max_depths = 7
# config.ds_alpha = 100.0 
config.ds_beta = 1.0
config.ds_alpha = 100.0
config.diversity_thres = 3
config.lam = 10
config.cpuct = 2
# config.num_phases = config.n**(config.max_depths-1)
config.num_batches = 10
config.batch_budget = config.num_batches*config.max_depths
# config.num_phases = 7
config.num_phases = config.batch_budget
print(config.batch_budget)
importlib.reload(mcts_search_mini_v55)
importlib.reload(mcts_search_mini_v15)
importlib.reload(mcts_search_mini_v35)
# importlib.reload(mcts_search_mini_v45)
# importlib.reload(mcts_search_mini_v31)
# importlib.reload(mcts_search_mini_v41)

70


<module 'core.mcts_search_mini_v35' from '/home/u20/tnguyen9210/tnn1/LLMs/llm-reasoning-methods/core/mcts_search_mini_v35.py'>

In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v35.MCTS(config=config, question=question)
    tree_dict_v35, tag_list_v35, completions_dict_v35 = mcts_search_mini_v35.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

config_name = f"mcts--v35--n-{config.n}--d-{config.max_depths}--nb-{config.num_batches}--lam-{config.lam}--dalpha-{config.ds_alpha}--dbeta-{config.ds_beta}--cpuct-{config.cpuct_root}-{config.cpuct_leaf}-{config.cpuct}--level-{level}--qidx-{q_idx}"

with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
    json.dump(completions_dict_v35, fout)
    fout.write('\n')

In [90]:
tags_list_v35 = set(completions_dict_v35.keys())
print(tags_list_v35)
print(f"num_tags = {len(completions_dict_v35)}")
for tag, text in completions_dict_v35.items():
    print(f"-> tag = {tag}, q_value = {tree_dict[tag]['q_value']:0.4f}")
    print(text)


{'0.2.1.3.3.3.0.0', '0.1.1.3.3.2.0', '0.1.3.1.1.2.3.0', '0.1.3.0.3.2.1', '0.3.0.2.0.1.0', '0.0.2.2.2.2.0', '0.1.3.0.3.2.0', '0.1.0.1.1.0', '0.0.1.3.3.2.3.0', '0.1.1.1.3.0', '0.0.0.3.2.1.0'}
num_tags = 11
-> tag = 0.1.3.1.1.2.3.0, q_value = 0.9912
## Step 1: Expand $(x+b)^2+c$ to compare with the given quadratic
First, we expand the expression $(x+b)^2+c$ to get $x^2+2bx+(b^2+c)$. We compare this with the given quadratic $x^2+(2.6)x+3.6$.

## Step 2: Match the coefficients of the expanded expression with the given quadratic
By matching the coefficients of the expanded expression $x^2+2bx+(b^2+c)$ with the given quadratic $x^2+(2.6)x+3.6$, we get two equations: $2b = 2.6$ and $b^2 + c = 3.6$.

## Step 3: Solve the equation $2b = 2.6$ for $b$
Solving the equation $2b = 2.6$ for $b$, we get $b = \frac{2.6}{2} = 1.3$.

## Step 4: Substitute the value of $b$ into the equation $b^2 + c = 3.6$ to solve for $c$
Substituting the value of $b = 1.3$ into the equation $b^2 + c = 3.6$, we get $(1.3)

In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v55.MCTS(config=config, question=question)
    tree_dict_v55, tag_list_v55, completions_dict_v55 = mcts_search_mini_v55.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

config_name = f"mcts--v55--n-{config.n}--d-{config.max_depths}--nb-{config.num_batches}--lam-{config.lam}--dalpha-{config.ds_alpha}--dbeta-{config.ds_beta}--cpuct-{config.cpuct_root}-{config.cpuct_leaf}-{config.cpuct}--level-{level}--qidx-{q_idx}"

with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
    json.dump(completions_dict_v55, fout)
    fout.write('\n')

In [75]:
tags_list_v55 = set(completions_dict_v55.keys())
print(tags_list_v55)
print(f"num_tags = {len(completions_dict_v55)}")
for tag, text in completions_dict_v55.items():
    print(f"-> tag = {tag}, q_value = {tree_dict[tag]['q_value']:0.4f}")
    # print(text)


{'0.1.0.3.0.1.1.0', '0.1.3.3.0.3.2.0', '0.3.2.2.1.0.2.1', '0.1.0.3.3.1.0', '0.1.0.1.0.0', '0.3.2.2.0.0.0', '0.3.3.0.3.0.0.1', '0.3.2.1.3.1.0'}
num_tags = 8
-> tag = 0.3.3.0.3.0.0.1, q_value = 0.9985
-> tag = 0.1.3.3.0.3.2.0, q_value = 0.1896
-> tag = 0.3.2.1.3.1.0, q_value = 0.9839
-> tag = 0.3.2.2.0.0.0, q_value = 0.9814
-> tag = 0.3.2.2.1.0.2.1, q_value = 0.9990
-> tag = 0.1.0.1.0.0, q_value = 0.9922
-> tag = 0.1.0.3.3.1.0, q_value = 1.0000
-> tag = 0.1.0.3.0.1.1.0, q_value = 0.9937


In [ ]:

for question in batch_of_questions:
    agent = mcts_search_mini_v15.MCTS(config=config, question=question)
    tree_dict_v15, tag_list_v15, completions_dict_v15 = mcts_search_mini_v15.mcts_search(question, agent, config, tree_dict)
    # print(agent_completions)

config_name = f"mcts--v15--n-{config.n}--d-{config.max_depths}--nb-{config.num_batches}--cpuct-{config.cpuct_root}-{config.cpuct_leaf}-{config.cpuct}--level-{level}--qidx-{q_idx}"
with open(f"results/sample_{config_name}--trial-{trial_idx}.jsonl", 'w', encoding = 'utf-8') as fout:
    json.dump(completions_dict_v15, fout)
    fout.write('\n')

In [77]:
tags_list_v15 = set(completions_dict_v15.keys())
print(tags_list_v15)
print(f"num_tags = {len(completions_dict_v15)}")
for tag, text in completions_dict_v15.items():
    print(f"tag = {tag}, q_value = {tree_dict[tag]['q_value']:0.4f}")
    # print(text)

{'0.0.3.3.2.0', '0.1.1.2.3.3.0', '0.1.3.3.0.3.2.0', '0.0.1.3.3.0.1.0', '0.0.0.3.2.3.0', '0.0.1.3.3.0.0', '0.0.0.0.1.1.3.0', '0.1.0.1.0.0', '0.3.3.0.3.0.0.1', '0.1.2.2.0.0.0', '0.0.1.2.3.0', '0.0.2.1.2.3.0', '0.3.2.1.3.1.0', '0.0.2.2.2.3.0', '0.0.3.1.0.1.0'}
num_tags = 15
tag = 0.1.3.3.0.3.2.0, q_value = 0.1896
tag = 0.3.3.0.3.0.0.1, q_value = 0.9985
tag = 0.0.2.2.2.3.0, q_value = 0.9771
tag = 0.0.0.3.2.3.0, q_value = 0.9917
tag = 0.0.1.2.3.0, q_value = 0.9985
tag = 0.0.3.1.0.1.0, q_value = 0.9976
tag = 0.0.2.1.2.3.0, q_value = 0.9873
tag = 0.3.2.1.3.1.0, q_value = 0.9839
tag = 0.0.0.0.1.1.3.0, q_value = 0.9878
tag = 0.0.1.3.3.0.0, q_value = 0.9985
tag = 0.0.1.3.3.0.1.0, q_value = 0.9976
tag = 0.1.1.2.3.3.0, q_value = 1.0000
tag = 0.1.2.2.0.0.0, q_value = 0.9961
tag = 0.0.3.3.2.0, q_value = 0.9976
tag = 0.1.0.1.0.0, q_value = 0.9922


In [78]:
tags_common = tags_list_v15 & tags_list_v55
tags_diff_v15 = tags_list_v15 - tags_common
tags_diff_v55 = tags_list_v55 - tags_common
print(tags_common)
print(tags_diff_v15)
print(tags_diff_v55)

{'0.3.3.0.3.0.0.1', '0.3.2.1.3.1.0', '0.1.3.3.0.3.2.0', '0.1.0.1.0.0'}
{'0.0.3.3.2.0', '0.1.1.2.3.3.0', '0.0.1.3.3.0.1.0', '0.0.0.3.2.3.0', '0.0.3.1.0.1.0', '0.0.0.0.1.1.3.0', '0.1.2.2.0.0.0', '0.0.1.2.3.0', '0.0.2.1.2.3.0', '0.0.2.2.2.3.0', '0.0.1.3.3.0.0'}
{'0.1.0.3.0.1.1.0', '0.1.0.3.3.1.0', '0.3.2.2.0.0.0', '0.3.2.2.1.0.2.1'}


In [87]:
tags_common = tags_list_v15 & tags_list_v35
tags_diff_v15 = tags_list_v15 - tags_common
tags_diff_v35 = tags_list_v35 - tags_common
print(tags_common)
print(tags_diff_v15)
print(tags_diff_v35)

set()
{'0.0.3.3.2.0', '0.1.1.2.3.3.0', '0.1.3.3.0.3.2.0', '0.0.1.3.3.0.1.0', '0.0.0.3.2.3.0', '0.0.1.3.3.0.0', '0.0.0.0.1.1.3.0', '0.1.0.1.0.0', '0.3.3.0.3.0.0.1', '0.1.2.2.0.0.0', '0.0.1.2.3.0', '0.0.2.1.2.3.0', '0.3.2.1.3.1.0', '0.0.2.2.2.3.0', '0.0.3.1.0.1.0'}
{'0.2.1.3.3.3.0.0', '0.1.1.3.3.2.0', '0.1.3.1.1.2.3.0', '0.1.3.0.3.2.1', '0.3.0.2.0.1.0', '0.0.2.2.2.2.0', '0.1.3.0.3.2.0', '0.1.0.1.1.0', '0.0.1.3.3.2.3.0', '0.1.1.1.3.0', '0.0.0.3.2.1.0'}


In [57]:
# print(tree_dict['0.1.3.3.0.3.2.0'])
# print(tree_dict['0.3.3.0.3.0.0.1'])
# print(tree_dict['0.0.2.2.2.3.0'])
print(tree_dict['0.0.3.1.0.1.0'])
# print(tree_dict['0.1']['text'])
# print(tree_dict['0.2']['text'])
# print(tree_dict['0.3']['text'])
# print(tree_dict['0.0.0.0.0.0.0.0.1'])
# print(tree_dict['0.0.0.0.0.0.0.0.2'])
print()
# print(tree_dict['0.1.3.2.1.0.1'])
# print(tree_dict['0.1.3.2.1.0.3'])

{'text': '## Step 1: Expand $(x+b)^2+c$ to match the quadratic $x^2+(2.6)x+3.6$\nWe start by expanding the expression $(x+b)^2+c$ using the formula $(a+b)^2 = a^2 + 2ab + b^2$. In our case, $a = x$ and $b = 2.6$. Therefore, $(x+b)^2+c = x^2 + 2bx + b^2 + c$.\n\n## Step 2: Match the coefficients of the quadratic terms\nMatching the coefficients of the quadratic terms, we get $b = 2.6$ and the constant term $c = b^2$.\n\n## Step 3: Calculate $b^2$\n$b^2 = (2.6)^2 = 6.76$\n\n## Step 4: Calculate $c$\n$c = b^2 = 6.76$\n\n## Step 5: Calculate $b + c$\n$b + c = 2.6 + 6.76 = 9.36$\n\nThe final answer is: $\\boxed{9.36}$', 'embeds': array([ 0.0106,  0.0095, -0.0271, ...,  0.0099,  0.0028, -0.0198],
      dtype=float32), 'q_value': 0.99755859375, 'is_terminal': True, 'is_completed': True}

